## 9. Limitação identificada: pacientes duplicados

O alerta **DUPLICATES** do ydata-profiling (285 linhas, 28,5%) foi investigado em mais detalhe. Esta seção mostra (9.1) quantos pacientes estão repetidos, (9.2) por que isso é um problema para a avaliação dos modelos e (9.3) como o banco ficaria sem as duplicatas. Para seguir a metodologia da disciplina, os modelos usam as 1000 linhas como vieram; as duplicatas são registradas aqui como **limitação** do projeto.

In [ ]:
# 9.1 Quantos pacientes estão repetidos?
# duplicated() marca como True cada linha que é cópia idêntica de uma anterior
copias_extras = df.duplicated().sum()
pacientes_unicos = len(df.drop_duplicates())

# Quantas vezes cada paciente aparece no banco
todas_colunas = list(df.columns)
vezes = df.groupby(todas_colunas, dropna=False).size()
repetidos = (vezes > 1).sum()

print(f"Linhas no banco: {len(df)}")
print(f"Pacientes únicos: {pacientes_unicos}")
print(f"Cópias extras (pandas, df.duplicated): {copias_extras}")
print(f"Pacientes que aparecem mais de uma vez (o '285' do ydata): {repetidos}")

pd.DataFrame(
    {"Pacientes": vezes.value_counts().sort_index()}
).rename_axis("Vezes que aparece no banco")

✍️ **Comentário:** o banco tem 1000 linhas, mas apenas **400 pacientes únicos**. Os dois números de duplicatas medem coisas diferentes: o ydata-profiling conta **285 pacientes que aparecem repetidos**, enquanto o `df.duplicated()` do pandas conta as **600 cópias extras**. Apenas 115 pacientes aparecem uma única vez; os demais aparecem de 2 a 8 vezes. As cópias são idênticas inclusive nos valores faltantes.

In [ ]:
# 9.2 Por que isso é um problema? Vazamento entre treino e teste
from sklearn.model_selection import train_test_split

# Divisão 70/15/15 estratificada SEM remover duplicatas (como estava antes)
treino_dup, resto_dup = train_test_split(
    df, test_size=0.30, stratify=df["classe"], random_state=42
)
valid_dup, teste_dup = train_test_split(
    resto_dup, test_size=0.50, stratify=resto_dup["classe"], random_state=42
)


def tem_copia_no_treino(parte, treino):
    # True para cada paciente da 'parte' que tem uma cópia idêntica no treino
    chave = lambda d: d.astype(str).agg("|".join, axis=1)  # noqa: E731
    return chave(parte).isin(set(chave(treino)))


vaz_teste = tem_copia_no_treino(teste_dup, treino_dup).sum()
vaz_valid = tem_copia_no_treino(valid_dup, treino_dup).sum()
print(f"Teste: {vaz_teste} de {len(teste_dup)} pacientes têm cópia idêntica no treino")
print(f"Validação: {vaz_valid} de {len(valid_dup)} pacientes têm cópia idêntica no treino")

✍️ **Comentário:** sem remover as duplicatas, **124 dos 150 pacientes do teste (83%)** e 115 dos 150 da validação têm uma cópia idêntica no treino. Isso é **vazamento de dados**: o modelo é avaliado em pacientes que ele já viu durante o treino, e as métricas ficam otimistas. Modelos que "decoram" os exemplos, como árvores de decisão e Random Forest, são os mais favorecidos por esse erro.

In [ ]:
# 9.3 Como ficaria o banco sem duplicatas? (análise; NÃO adotada nos modelos)
import sys

sys.path.append("..")  # permite importar a pasta src a partir de notebooks/
from scipy.stats import chisquare
from src.dados import dividir_dados, estruturar_dados, remover_duplicatas

# Nomes novos (sufixo _unico) para não sobrescrever variáveis das seções anteriores
df_unico = remover_duplicatas(df)

# Balanceamento
contagem_unico = df_unico["classe"].value_counts()
_, p_unico = chisquare(contagem_unico)
print(f"Pacientes únicos por classe: {contagem_unico.to_dict()}")
print(f"Qui-quadrado de aderência à distribuição uniforme: p = {p_unico:.3f}")

# Divisão que se obteria removendo as duplicatas antes de dividir
treino_unico, validacao_unico, teste_unico = dividir_dados(estruturar_dados(df_unico))
juntos_unico = pd.concat([treino_unico, validacao_unico, teste_unico])
print(
    f"Treino: {len(treino_unico)} | Validação: {len(validacao_unico)} "
    f"| Teste: {len(teste_unico)}"
)
print(f"Pacientes repetidos entre as partes: {juntos_unico.duplicated().sum()}")

# Faltantes depois da remoção
falt_unico = df_unico.isna().sum()
falt_unico = falt_unico[falt_unico > 0].sort_values(ascending=False)
pd.DataFrame(
    {
        "Faltantes": falt_unico,
        "Proporção (%)": (falt_unico / len(df_unico) * 100).round(1),
    }
)

✍️ **Comentário:** sem as duplicatas, o banco ficaria com 400 pacientes, **exatamente 100 por classe** (balanceamento perfeito, qui-quadrado p = 1,000), e a divisão estratificada teria 280 pacientes no treino, 60 na validação e 60 no teste, sem nenhum paciente repetido entre as partes. Os faltantes continuariam nas mesmas seis variáveis (PPD com 7,5%, 30 de 400). Essa alternativa está disponível no código pela função `remover_duplicatas` (`src/dados.py`), mas **não foi adotada nos modelos**: para seguir a metodologia da disciplina, o `src/pipeline.py` usa as 1000 linhas, com 700 no treino, 150 na validação e 150 no teste.

### Conclusão final da AED

- O banco tem 1000 linhas, mas apenas **400 pacientes únicos** (100 por classe): 600 linhas são cópias. Para seguir a metodologia da disciplina, os modelos usam as 1000 linhas como vieram (700 treino, 150 validação, 150 teste).
- **Limitação:** como 124 dos 150 pacientes do teste têm cópia idêntica no treino, as métricas dos modelos devem ser interpretadas como **otimistas**. Modelos que "decoram" exemplos, como árvores de decisão e Random Forest, são os mais favorecidos. Em uma aplicação real, as duplicatas deveriam ser removidas antes da divisão (função `remover_duplicatas`).
- As demais decisões se mantêm: remoção das constantes, transformação log da PPD, redução do bloco correlacionado ao CD7 e as variáveis selecionadas (TP53, IFN-γ, CD3_2, CD8, PPD_log, CD7, FOXP3).
- Hipótese para a modelagem: alto acerto para BCG e imunodeficiência e dificuldade em separar cancer de leucemia, que não se distinguem por nenhum dos biomarcadores selecionados.